# End-to-End ML Project — Manual Version (No Pipeline / ColumnTransformer)

**Goal:** Predict whether a student **PASSES (1)** or **FAILS (0)** based on study habits, attendance, demographics, and academic history.

Every preprocessing step below (imputation, encoding, scaling) is done **by hand** with pandas/numpy instead of `sklearn.pipeline.Pipeline` or `sklearn.compose.ColumnTransformer`, so you can see exactly what those tools normally do under the hood.

**Golden rule followed throughout:** split train/test **first**, then *fit* every preprocessing step (imputer medians, category lists, scaler mean/std) **on the training set only**, and *apply* those same learned values to the test set and to any new data later. This avoids data leakage.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

pd.set_option("display.width", 120)

## Step 0 — Raw Data

In [2]:
df = pd.DataFrame({
    "Study_Hours":[5,8,np.nan,2,9,7,6,4,10,3,8,6],
    "Attendance":[90,95,85,np.nan,98,92,88,75,99,80,94,87],
    "Gender":["Male","Female","Female","Male","Female",
              "Male","Male","Female","Female","Male","Female","Male"],
    "City":["Dhaka","Dhaka","Chittagong","Khulna","Dhaka",
            "Khulna","Rajshahi","Dhaka","Rajshahi",
            "Chittagong","Dhaka","Khulna"],
    "Family_Income":[50000,70000,65000,40000,np.nan,
                     80000,55000,48000,90000,45000,72000,60000],
    "Previous_GPA":[3.2,3.8,3.5,2.7,3.9,3.4,3.1,np.nan,4.0,2.9,3.7,3.3],
    "Pass":[1,1,1,0,1,1,1,0,1,0,1,1]
})
df

,Study_Hours,Attendance,Gender,City,Family_Income,Previous_GPA,Pass
0,5.0,90.0,Male,Dhaka,50000.0,3.2,1
1,8.0,95.0,Female,Dhaka,70000.0,3.8,1
2,NaN,85.0,Female,Chittagong,65000.0,3.5,1
3,2.0,NaN,Male,Khulna,40000.0,2.7,0
4,9.0,98.0,Female,Dhaka,NaN,3.9,1
5,7.0,92.0,Male,Khulna,80000.0,3.4,1
6,6.0,88.0,Male,Rajshahi,55000.0,3.1,1
7,4.0,75.0,Female,Dhaka,48000.0,NaN,0
8,10.0,99.0,Female,Rajshahi,90000.0,4.0,1
9,3.0,80.0,Male,Chittagong,45000.0,2.9,0


## Step 1 — Exploratory Data Analysis (EDA)

In [3]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)

Shape: (12, 7)

Dtypes:
 Study_Hours      float64
Attendance       float64
Gender               str
City                 str
Family_Income    float64
Previous_GPA     float64
Pass               int64
dtype: object


In [4]:
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 Study_Hours      1
Attendance       1
Gender           0
City             0
Family_Income    1
Previous_GPA     1
Pass             0
dtype: int64


In [5]:
df.describe()

,Study_Hours,Attendance,Family_Income,Previous_GPA,Pass
count,11.000000,11.000000,11.000000,11.000000,12.000000
mean,6.181818,89.363636,61363.636364,3.409091,0.750000
std,2.522625,7.406385,15628.645029,0.418221,0.452267
min,2.000000,75.000000,40000.000000,2.700000,0.000000
25%,4.500000,86.000000,49000.000000,3.150000,0.750000
50%,6.000000,90.000000,60000.000000,3.400000,1.000000
75%,8.000000,94.500000,71000.000000,3.750000,1.000000
max,10.000000,99.000000,90000.000000,4.000000,1.000000


In [6]:
print("Target balance:\n", df["Pass"].value_counts())

Target balance:
 Pass
1    9
0    3
Name: count, dtype: int64


## Step 2 — Train / Test Split (BEFORE any preprocessing!)

**Critical rule:** split first, then learn imputation values / scaling stats / encoding categories **only from the training set**. This avoids "data leakage" from the test set into training decisions.

In [7]:
X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
# work on copies so we never touch the originals by accident
X_train = X_train.copy()
X_test = X_test.copy()

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

Train shape: (9, 6)  Test shape: (3, 6)


## Step 3 — Missing Value Imputation (manual, fit on TRAIN only)

In [8]:
numeric_cols = ["Study_Hours", "Attendance", "Family_Income", "Previous_GPA"]

# "Fit": learn the median of each numeric column FROM TRAINING DATA ONLY
impute_values = {col: X_train[col].median() for col in numeric_cols}
print("Median values learned from TRAIN set (used to fill NaNs):")
for k, v in impute_values.items():
    print(f"  {k}: {v}")

Median values learned from TRAIN set (used to fill NaNs):
  Study_Hours: 6.0
  Attendance: 89.0
  Family_Income: 57500.0
  Previous_GPA: 3.4


In [9]:
# "Transform": apply the SAME learned values to both train and test
for col in numeric_cols:
    X_train[col] = X_train[col].fillna(impute_values[col])
    X_test[col] = X_test[col].fillna(impute_values[col])

print("Missing values after imputation (train):\n", X_train[numeric_cols].isnull().sum())
print("\nMissing values after imputation (test):\n", X_test[numeric_cols].isnull().sum())

Missing values after imputation (train):
 Study_Hours      0
Attendance       0
Family_Income    0
Previous_GPA     0
dtype: int64

Missing values after imputation (test):
 Study_Hours      0
Attendance       0
Family_Income    0
Previous_GPA     0
dtype: int64


## Step 4 — Encoding Categorical Variables (manual)

- **Gender**: binary column → simple manual mapping (label encoding)
- **City**: nominal, >2 categories → manual one-hot encoding, using a category list *fixed from the training set* so train/test always end up with the same dummy columns in the same order

In [10]:
gender_map = {"Male": 0, "Female": 1}
X_train["Gender"] = X_train["Gender"].map(gender_map)
X_test["Gender"] = X_test["Gender"].map(gender_map)
print("Gender encoded with mapping:", gender_map)

Gender encoded with mapping: {'Male': 0, 'Female': 1}


In [11]:
# "Fit": learn the set of known categories from TRAIN only
city_categories = sorted(X_train["City"].unique())
print("City categories learned from TRAIN:", city_categories)

def one_hot_encode(df_in, column, categories):
    """Manually one-hot encode `column` using a FIXED category list
    (learned from training data), so train/test always end up with
    the exact same set of dummy columns in the same order."""
    out = df_in.copy()
    for cat in categories:
        out[f"{column}_{cat}"] = (out[column] == cat).astype(int)
    out = out.drop(columns=[column])
    return out

X_train = one_hot_encode(X_train, "City", city_categories)
X_test = one_hot_encode(X_test, "City", city_categories)

# Note: if the test set had an unseen category, one_hot_encode would just
# produce all-zero dummy columns for that row -> which is correct behavior.

print("Columns after encoding:\n", X_train.columns.tolist())

City categories learned from TRAIN: ['Chittagong', 'Dhaka', 'Khulna', 'Rajshahi']
Columns after encoding:
 ['Study_Hours', 'Attendance', 'Gender', 'Family_Income', 'Previous_GPA', 'City_Chittagong', 'City_Dhaka', 'City_Khulna', 'City_Rajshahi']


## Step 5 — Feature Scaling (manual standardization, fit on TRAIN only)

In [12]:
scale_cols = ["Study_Hours", "Attendance", "Family_Income", "Previous_GPA"]

# "Fit": learn mean & std from TRAIN only
scale_params = {
    col: {"mean": X_train[col].mean(), "std": X_train[col].std()}
    for col in scale_cols
}
print("Mean/Std learned from TRAIN set:")
for k, v in scale_params.items():
    print(f"  {k}: mean={v['mean']:.2f}, std={v['std']:.2f}")

Mean/Std learned from TRAIN set:
  Study_Hours: mean=6.22, std=2.49
  Attendance: mean=89.56, std=7.35
  Family_Income: mean=59500.00, std=14568.80
  Previous_GPA: mean=3.43, std=0.42


In [13]:
# "Transform": apply to both train and test using TRAIN's mean/std
for col in scale_cols:
    mean, std = scale_params[col]["mean"], scale_params[col]["std"]
    X_train[col] = (X_train[col] - mean) / std
    X_test[col] = (X_test[col] - mean) / std

X_train.head()

,Study_Hours,Attendance,Gender,Family_Income,Previous_GPA,City_Chittagong,City_Dhaka,City_Khulna,City_Rajshahi
3,-1.696445,-0.075582,0,-1.338477,-1.753002,0,0,1,0
8,1.517872,1.284896,1,2.093515,1.354592,0,0,0,1
2,-0.089287,-0.619773,1,0.377519,0.159364,1,0,0,0
7,-0.892866,-1.980251,1,-0.789358,-0.079682,0,1,0,0
11,-0.089287,-0.347678,0,0.034320,-0.318728,0,0,1,0


## Step 6 — Model Training (Logistic Regression)

In [14]:
# Make sure train/test have identical column order
X_test = X_test[X_train.columns]

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("Model trained.")
print("Feature order used:", X_train.columns.tolist())
print("Learned coefficients:", np.round(model.coef_[0], 3))
print("Intercept:", round(model.intercept_[0], 3))

Model trained.
Feature order used: ['Study_Hours', 'Attendance', 'Gender', 'Family_Income', 'Previous_GPA', 'City_Chittagong', 'City_Dhaka', 'City_Khulna', 'City_Rajshahi']
Learned coefficients: [ 0.776  0.654 -0.205  0.604  0.37   0.12  -0.101 -0.172  0.153]
Intercept: 2.268


## Step 7 — Evaluation on Test Set

In [15]:
y_pred = model.predict(X_test)

print("Predictions: ", y_pred.tolist())
print("Actual:      ", y_test.tolist())

print("\nAccuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1-score :", round(f1_score(y_test, y_pred, zero_division=0), 3))

Predictions:  [0, 1, 1]
Actual:       [0, 1, 1]

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0


In [16]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Confusion Matrix:
 [[1 0]
 [0 2]]


In [17]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



> **Mentor's note:** with only 12 rows total, this dataset is purely for learning the *mechanics*. A perfect test score here is an artifact of a 3-row test set — it doesn't mean the model would generalize on real, larger data.

## Step 8 — Inference on a New, Unseen Student

This rebuilds the entire manual pipeline step-by-step on a single new row — including a missing value and a city ("Sylhet") that was **never seen during training** — to prove the "fit on train, reuse everywhere" pattern actually generalizes.

In [18]:
new_student = pd.DataFrame({
    "Study_Hours": [7],
    "Attendance": [np.nan],       # missing on purpose, to test imputation
    "Gender": ["Female"],
    "City": ["Sylhet"],           # unseen category on purpose
    "Family_Income": [58000],
    "Previous_GPA": [3.4],
})

# 1) impute using the SAME learned train medians
for col in numeric_cols:
    new_student[col] = new_student[col].fillna(impute_values[col])

# 2) encode using the SAME learned mappings
new_student["Gender"] = new_student["Gender"].map(gender_map)
new_student = one_hot_encode(new_student, "City", city_categories)

# 3) scale using the SAME learned train mean/std
for col in scale_cols:
    mean, std = scale_params[col]["mean"], scale_params[col]["std"]
    new_student[col] = (new_student[col] - mean) / std

# 4) align columns exactly like training data
new_student = new_student.reindex(columns=X_train.columns, fill_value=0)

new_student

,Study_Hours,Attendance,Gender,Family_Income,Previous_GPA,City_Chittagong,City_Dhaka,City_Khulna,City_Rajshahi
0,0.312503,-0.075582,1,-0.10296,-0.079682,0,0,0,0


In [19]:
pred = model.predict(new_student)[0]
prob = model.predict_proba(new_student)[0][1]

print(f"Prediction: {'PASS' if pred==1 else 'FAIL'} (probability of passing = {prob:.2f})")

Prediction: PASS (probability of passing = 0.90)


## Next Steps

- Try `RandomForestClassifier` and compare metrics against Logistic Regression
- Add k-fold cross-validation instead of a single train/test split (this dataset is tiny — 12 rows — so a single split is noisy)
- Try `KNNImputer` instead of median-fill for missing values
- Once you're comfortable with each manual step, revisit `Pipeline` + `ColumnTransformer` — you'll now know exactly what they're automating